In [9]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain.tools import tool
load_dotenv()

if os.environ['GOOGLE_API_KEY'] == "":
    raise ValueError("GOOGLE_API_KEY is not set in the .env file.")
else:
    print("GOOGLE_API_KEY is set correctly.")

GOOGLE_API_KEY is set correctly.


In [10]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash",)
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.5-flash', temperature=1.0, client=<google.genai.client.Client object at 0x000001CCF64CCFA0>, default_metadata=(), model_kwargs={})

In [21]:
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def search_trending_news(topic: str):
    """Use this tool to search for latest news on DuckDuckGo for a given topic and returns the results."""
    search = DuckDuckGoSearchResults()
    return search.invoke(f"Latest news on {topic}?")

In [14]:
from langchain_community.retrievers import ArxivRetriever

@tool
def arxiv_retriever(query: str):
    """Use this tool to retrieve documents from arXiv based on a query."""
    retriever = ArxivRetriever( load_max_docs=2, get_ful_documents=True, )
    return retriever.invoke(query)

In [15]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wikipedia_query(query: str):
    """Use this tool to query Wikipedia for information."""
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wikipedia.invoke(query)

# **Custom Tool**

In [18]:
@tool
def personal_info(name: str):
    """Use this tool to get personal information about Alice, Bob or Charlie."""
    info = {
        "Alice": "Alice is a software engineer from San Francisco.",
        "Bob": "Bob is a data scientist from New York.",
        "Charlie": "Charlie is a product manager from Seattle.",
    }
    return info.get(name, "Person not found.")

# **Tool binding**

In [22]:
tools = [search_trending_news, arxiv_retriever, wikipedia_query, personal_info]

llm_with_tools = llm.bind_tools(tools)

In [24]:
response = llm_with_tools.invoke("Can you tell me the latest news on AI?")

In [25]:
response.tool_calls

[{'name': 'search_trending_news',
  'args': {'topic': 'Artificial Intelligence'},
  'id': 'smlngfv2',
  'type': 'tool_call'}]